In [ ]:
# =============================================================
# 04_descriptive_statistics.ipynb
# Step 4
# ------------------------------------------------------------
# What this does
# ──────────────
# Reads the wide panel and trajectory labels from Steps 2–3
# and produces all descriptive tables and figures for §4 of
# the manuscript:
#
# Tables (CSV + LaTeX)
# ─────────────────────
#   Table 1 — Panel completeness by income group and indicator
#             (proportion of 10 years with observed data,
#              pre- and post-interpolation)
#
#   Table 2 — Pearson correlation matrix across 17 indicators
#             (complete-case, n per pair logged)
#
#   Table 3 — Trajectory delta summary
#             (ΔGini, ΔTertiary, ΔCompletion per country,
#              with trajectory class)
#
# Figures
# ────────
#   Fig 1 — MNAR pattern bar chart
#           (missing % per indicator, coloured by income tier)
#
#   Fig 2 — Correlation heatmap (lower triangle, annotated)
#
# Usage
# ─────
#   python src/04_descriptive_statistics.py
#
# Requirements
# ────────────
#   pip install pandas numpy matplotlib seaborn pyyaml
# =============================================================

from __future__ import annotations

import sys
from pathlib import Path

try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    _SRC = _cwd / "src" if (_cwd / "src").exists() else _cwd
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from utils import find_project_root, load_indicators, load_pipeline, INCOME_ORDER


# ──────────────────────────────────────────────────────────────
# Setup
# ──────────────────────────────────────────────────────────────
PROJECT_ROOT = find_project_root()
IND_CFG      = load_indicators()
PIPE_CFG     = load_pipeline()

YEAR_MIN   = PIPE_CFG["years"]["min"]
YEAR_MAX   = PIPE_CFG["years"]["max"]
PERIOD_TAG = f"{YEAR_MIN}_{YEAR_MAX}"

ALL_INDICATORS = list(IND_CFG["indicators"].keys())

# Indicator display labels (short form for tables/figures)
IND_LABELS = {
    "gini":             "Gini index",
    "inc_share_low20":  "Income share (low 20%)",
    "poverty_215":      "Poverty ($2.15/day)",
    "sec_enrol_gross":  "Sec. net enrolment",
    "tert_enrol_gross": "Tert. gross enrolment",
    "sec_completion":   "Sec. completion rate",
    "adult_literacy":   "Adult literacy (15+)",
    "youth_literacy":   "Youth literacy (15–24)",
    "gpi_sec":          "GPI secondary",
    "gpi_tert":         "GPI tertiary",
    "edu_spend_gdp":    "Edu. spend (% GDP)",
    "gdp_pc_ppp":       "GDP/capita (PPP)",
    "pop_total":        "Population",
    "work_age_share":   "Working-age share",
    "gov_effect":       "Govt. effectiveness",
    "voice_account":    "Voice & accountability",
    "ctrl_corrup":      "Control of corruption",
}

# Roles for manuscript notation
IND_ROLES = {k: v.get("role","X") for k,v in IND_CFG["indicators"].items()}

DATA_DIR  = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGDIR    = PROJECT_ROOT / "outputs" / "figures" / "panel_12"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGDIR.mkdir(parents=True, exist_ok=True)

print(f"[info] Project root  : {PROJECT_ROOT}")
print(f"[info] Indicators    : {len(ALL_INDICATORS)}")


# ──────────────────────────────────────────────────────────────
# Load data
# ──────────────────────────────────────────────────────────────
def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    panel_path = DATA_DIR / f"panel_12countries_{PERIOD_TAG}.csv"
    mask_path  = DATA_DIR / f"interpolation_mask_{PERIOD_TAG}.csv"
    label_path = DATA_DIR / "trajectory_labels.csv"

    for p in [panel_path, mask_path, label_path]:
        if not p.exists():
            raise FileNotFoundError(
                f"Required file not found: {p}\n"
                "Run steps 02 and 03 first."
            )

    panel  = pd.read_csv(panel_path)
    mask   = pd.read_csv(mask_path)
    labels = pd.read_csv(label_path)

    print(f"\n[step 0] Loaded:")
    print(f"  panel  : {len(panel)} rows × {len(panel.columns)} cols")
    print(f"  mask   : {len(mask)} rows × {len(mask.columns)} cols")
    print(f"  labels : {len(labels)} countries")
    return panel, mask, labels


# ──────────────────────────────────────────────────────────────
# Table 1 — Panel completeness
# ──────────────────────────────────────────────────────────────
def build_table1(panel: pd.DataFrame, mask: pd.DataFrame) -> pd.DataFrame:
    """
    Per-indicator completeness:
      - observed_pct   : fraction of 120 cells with M_it = 1 (empirically observed)
      - available_pct  : fraction with non-null value after interpolation
      - countries_miss : ISO codes of countries with ANY missing post-interpolation
    """
    ind_cols = [c for c in ALL_INDICATORS if c in panel.columns]
    N        = len(panel)   # 120

    records = []
    for nice in ind_cols:
        observed   = int(mask[nice].sum()) if nice in mask.columns else None
        available  = int(panel[nice].notna().sum())
        c_miss     = sorted(panel.loc[panel[nice].isna(), "iso3c"].unique())

        records.append({
            "indicator":       nice,
            "label":           IND_LABELS.get(nice, nice),
            "role":            IND_ROLES.get(nice, "X"),
            "observed_n":      observed,
            "observed_pct":    round(observed / N * 100, 1) if observed is not None else None,
            "available_n":     available,
            "available_pct":   round(available / N * 100, 1),
            "missing_n":       N - available,
            "missing_pct":     round((N - available) / N * 100, 1),
            "countries_miss":  ", ".join(c_miss) if c_miss else "—",
        })

    t1 = pd.DataFrame(records)
    # Sort by role then missing_pct descending
    role_order = {"Y": 0, "W": 1, "A": 2, "X": 3, "C": 4}
    t1["_rsort"] = t1["role"].map(role_order).fillna(5)
    t1 = t1.sort_values(["_rsort", "missing_pct"], ascending=[True, False])
    t1 = t1.drop(columns="_rsort").reset_index(drop=True)
    return t1


def table1_to_latex(t1: pd.DataFrame) -> str:
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Panel completeness by indicator (120 country-year cells, "
        r"2015--2024). \emph{Observed} = empirically recorded (M\textsubscript{it}=1); "
        r"\emph{Available} = non-null after linear interpolation of gaps "
        r"$\leq 2$ consecutive years.}",
        r"\label{tab:completeness}",
        r"\begin{tabular}{llccrr}",
        r"\toprule",
        r"Indicator & Role & Observed (\%) & Available (\%) & Missing $n$ "
        r"& Countries missing \\",
        r"\midrule",
    ]
    for _, row in t1.iterrows():
        obs  = f"{row['observed_pct']:.1f}" if pd.notna(row['observed_pct']) else "--"
        avail = f"{row['available_pct']:.1f}"
        miss  = str(int(row['missing_n']))
        clist = row['countries_miss']
        # Escape underscores in label
        label = row['label'].replace("_", r"\_")
        lines.append(
            f"{label} & {row['role']} & {obs} & {avail} & {miss} "
            f"& \\footnotesize{{{clist}}} \\\\"
        )
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
    return "\n".join(lines)


# ──────────────────────────────────────────────────────────────
# Table 2 — Pearson correlation matrix
# ──────────────────────────────────────────────────────────────
def build_table2(panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (corr_matrix, n_matrix) — both indexed by indicator nice names.
    n_matrix[i,j] = number of complete-case pairs used for r_ij.
    """
    ind_cols = [c for c in ALL_INDICATORS if c in panel.columns]
    data     = panel[ind_cols]

    n_ind  = len(ind_cols)
    corr   = np.full((n_ind, n_ind), np.nan)
    n_mat  = np.zeros((n_ind, n_ind), dtype=int)

    for i in range(n_ind):
        for j in range(n_ind):
            xy  = data[[ind_cols[i], ind_cols[j]]].dropna()
            n   = len(xy)
            n_mat[i, j] = n
            if n >= 3:
                corr[i, j] = float(xy.iloc[:, 0].corr(xy.iloc[:, 1]))

    corr_df = pd.DataFrame(corr,   index=ind_cols, columns=ind_cols)
    n_df    = pd.DataFrame(n_mat,  index=ind_cols, columns=ind_cols)
    return corr_df, n_df


def table2_to_latex(corr_df: pd.DataFrame, n_df: pd.DataFrame) -> str:
    """Lower-triangle LaTeX correlation table with significance markers."""
    cols  = list(corr_df.columns)
    n_col = len(cols)
    short = [IND_LABELS.get(c, c)[:18] for c in cols]   # truncate for table width

    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\tiny",
        r"\caption{Pearson correlation matrix (complete-case pairwise $n$ "
        r"varies by pair; lower triangle shown).}",
        r"\label{tab:correlation}",
        r"\begin{tabular}{l" + "r" * n_col + "}",
        r"\toprule",
        " & " + " & ".join([f"({i+1})" for i in range(n_col)]) + r" \\",
        r"\midrule",
    ]
    for i, col_i in enumerate(cols):
        row_label = f"({i+1}) {short[i]}"
        cells = []
        for j in range(n_col):
            if j > i:
                cells.append("")
            elif j == i:
                cells.append("1.00")
            else:
                r = corr_df.iloc[i, j]
                n = n_df.iloc[i, j]
                if np.isnan(r):
                    cells.append("--")
                else:
                    sig = ""
                    if n >= 3:
                        from scipy import stats as _stats
                        t_stat = r * np.sqrt(n - 2) / np.sqrt(1 - r**2 + 1e-12)
                        p = 2 * _stats.t.sf(abs(t_stat), df=n - 2)
                        if p < 0.001: sig = "***"
                        elif p < 0.01: sig = "**"
                        elif p < 0.05: sig = "*"
                    cells.append(f"{r:.2f}{sig}")
        lines.append(row_label + " & " + " & ".join(cells) + r" \\")
    lines += [
        r"\bottomrule",
        r"\multicolumn{" + str(n_col + 1) + r"}{l}{\footnotesize "
        r"$^{***}p<0.001$, $^{**}p<0.01$, $^{*}p<0.05$} \\",
        r"\end{tabular}",
        r"\end{table}",
    ]
    return "\n".join(lines)


# ──────────────────────────────────────────────────────────────
# Table 3 — Trajectory delta summary
# ──────────────────────────────────────────────────────────────
def build_table3(labels: pd.DataFrame) -> pd.DataFrame:
    cols = ["iso3c", "country", "income_group", "label_code", "label_name",
            "delta_gini", "delta_tert", "delta_comp",
            "gini_coverage", "label_reason"]
    t3 = labels[[c for c in cols if c in labels.columns]].copy()
    t3 = (
        t3.assign(_isort=lambda d: d["income_group"].map(INCOME_ORDER).fillna(4))
        .sort_values(["_isort", "label_code"])
        .drop(columns="_isort")
        .reset_index(drop=True)
    )
    return t3


def table3_to_latex(t3: pd.DataFrame) -> str:
    CLASS_COLOURS_TEX = {
        "EE": r"\cellcolor{green!15}",
        "AE": r"\cellcolor{blue!10}",
        "IR": r"\cellcolor{red!10}",
        "DL": r"\cellcolor{gray!15}",
    }
    lines = [
        r"\usepackage{colortbl}",   # reminder — add to preamble
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Trajectory classification: decade-long deltas and class "
        r"assignments. EE=Equity Expander, AE=Access Expander, "
        r"IR=Inequality Reducer, DL=Data-Limited. "
        r"$\Delta$Gini and $\Delta$Tertiary in percentage points "
        r"(first to last observed value, 2015--2024).}",
        r"\label{tab:trajectory}",
        r"\begin{tabular}{llllrrrc}",
        r"\toprule",
        r"ISO & Country & Tier & Class & $\Delta$Gini & $\Delta$Tert & "
        r"$\Delta$Comp & Gini cov. \\",
        r"\midrule",
    ]
    for _, row in t3.iterrows():
        shade = CLASS_COLOURS_TEX.get(row["label_code"], "")
        tier  = (row["income_group"]
                 .replace("Upper middle income", "Upper mid.")
                 .replace("Lower middle income", "Lower mid.")
                 .replace("High income", "High")
                 .replace("Low income", "Low"))
        dg = f"{row['delta_gini']:+.1f}" if pd.notna(row["delta_gini"]) else "--"
        dt = f"{row['delta_tert']:+.1f}" if pd.notna(row["delta_tert"]) else "--"
        dc = f"{row['delta_comp']:+.1f}" if pd.notna(row["delta_comp"]) else "--"
        gc = f"{row['gini_coverage']:.0%}"
        lines.append(
            f"{shade}{row['iso3c']} & {row['country']} & {tier} & "
            f"{row['label_code']} & {dg} & {dt} & {dc} & {gc} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


# ──────────────────────────────────────────────────────────────
# Figure 1 — MNAR missingness bar chart
# ──────────────────────────────────────────────────────────────
def plot_mnar_chart(panel: pd.DataFrame) -> None:
    """
    Horizontal bar chart: missing % per indicator,
    bars coloured by worst income tier affected.
    Annotated with missing country codes.
    """
    ind_cols = [c for c in ALL_INDICATORS if c in panel.columns]
    N        = len(panel)

    TIER_COLOURS = {
        "High income":         "#4393c3",
        "Upper middle income": "#74c476",
        "Lower middle income": "#fd8d3c",
        "Low income":          "#9e0142",
        "none":                "#cccccc",
    }
    TIER_ORDER_REV = ["Low income", "Lower middle income",
                      "Upper middle income", "High income", "none"]

    records = []
    for nice in ind_cols:
        miss_rows = panel[panel[nice].isna()]
        pct = miss_rows.shape[0] / N * 100
        if miss_rows.empty:
            worst_tier = "none"
        else:
            tier_order = {t: i for i, t in enumerate(TIER_ORDER_REV)}
            worst_tier = (
                miss_rows["income_group"]
                .map(lambda t: (tier_order.get(t, 99), t))
                .min()[1]
            )
        c_miss = sorted(miss_rows["iso3c"].unique())
        records.append({
            "nice":      nice,
            "label":     IND_LABELS.get(nice, nice),
            "pct":       pct,
            "worst":     worst_tier,
            "countries": ", ".join(c_miss),
        })

    df = pd.DataFrame(records).sort_values("pct", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 7))

    bars = ax.barh(
        df["label"], df["pct"],
        color=[TIER_COLOURS.get(t, "#cccccc") for t in df["worst"]],
        edgecolor="white", linewidth=0.5, height=0.65,
    )

    # Annotate with country codes
    for i, (_, row) in enumerate(df.iterrows()):
        if row["pct"] > 0:
            ax.text(
                row["pct"] + 0.5, i,
                row["countries"],
                va="center", ha="left", fontsize=7.5, color="#444444",
            )

    # Legend
    legend_handles = [
        mpatches.Patch(color=TIER_COLOURS[t], label=t)
        for t in ["Low income", "Lower middle income",
                  "Upper middle income", "High income"]
    ]
    legend_handles.append(
        mpatches.Patch(color=TIER_COLOURS["none"], label="No missing data")
    )
    ax.legend(handles=legend_handles, title="Worst tier affected",
              fontsize=8, title_fontsize=8,
              loc="lower right", frameon=True, framealpha=0.92,
              edgecolor="#cccccc")

    ax.set_xlabel("Missing data (%)", fontsize=11)
    ax.set_title(
        f"Indicator missingness — 12-country panel ({YEAR_MIN}–{YEAR_MAX})\n"
        "(post-interpolation; colour = lowest income tier with missing data)",
        fontsize=11, fontweight="bold",
    )
    ax.set_xlim(0, 55)
    ax.axvline(0, color="#888888", lw=0.6)
    ax.grid(axis="x", alpha=0.25, lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    out = FIGDIR / f"mnar_missingness_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


# ──────────────────────────────────────────────────────────────
# Figure 2 — Correlation heatmap
# ──────────────────────────────────────────────────────────────
def plot_correlation_heatmap(corr_df: pd.DataFrame, n_df: pd.DataFrame) -> None:
    """
    Lower-triangle annotated heatmap.
    Cells with n < 10 complete-case pairs shown in grey.
    """
    ind_cols = list(corr_df.columns)
    n_ind    = len(ind_cols)
    labels   = [IND_LABELS.get(c, c) for c in ind_cols]

    # Mask upper triangle
    mask = np.triu(np.ones((n_ind, n_ind), dtype=bool), k=1)
    corr_plot = corr_df.values.copy()
    corr_plot[mask] = np.nan

    fig, ax = plt.subplots(figsize=(13, 11))

    # Custom diverging colormap
    import matplotlib.colors as mcolors
    cmap = plt.cm.RdYlBu_r

    im = ax.imshow(corr_plot, aspect="auto", interpolation="nearest",
                   vmin=-1, vmax=1, cmap=cmap)

    # Annotate cells
    for i in range(n_ind):
        for j in range(n_ind):
            if j > i:
                continue
            r = corr_df.iloc[i, j]
            n = n_df.iloc[i, j]
            if np.isnan(r):
                continue
            if n < 10:
                # Low-n: grey background, smaller text
                ax.add_patch(plt.Rectangle(
                    (j - 0.5, i - 0.5), 1, 1,
                    color="#e0e0e0", zorder=2,
                ))
                ax.text(j, i, f"{r:.2f}\n(n={n})", ha="center", va="center",
                        fontsize=5.5, color="#666666", zorder=3)
            else:
                txt_color = "white" if abs(r) > 0.65 else "black"
                ax.text(j, i, f"{r:.2f}", ha="center", va="center",
                        fontsize=6.5, color=txt_color, zorder=3)

    ax.set_xticks(range(n_ind))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(n_ind))
    ax.set_yticklabels(labels, fontsize=8)

    cbar = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label("Pearson r", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    ax.set_title(
        f"Pearson correlation matrix — 12-country panel ({YEAR_MIN}–{YEAR_MAX})\n"
        "(complete-case pairwise; grey = n < 10 pairs)",
        fontsize=11, fontweight="bold",
    )
    ax.spines[["top", "right", "bottom", "left"]].set_visible(False)
    fig.tight_layout()

    out = FIGDIR / f"correlation_heatmap_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


# ──────────────────────────────────────────────────────────────
# Save tables
# ──────────────────────────────────────────────────────────────
def save_tables(
    t1: pd.DataFrame,
    corr_df: pd.DataFrame,
    n_df: pd.DataFrame,
    t3: pd.DataFrame,
) -> None:
    print("\n[step 5] Saving tables …")

    # Table 1
    t1.to_csv(TABLE_DIR / f"table1_completeness.csv", index=False)
    (TABLE_DIR / f"table1_completeness.tex").write_text(
        table1_to_latex(t1), encoding="utf-8"
    )
    print(f"  [OK] table1_completeness.csv / .tex")

    # Table 2
    corr_df.to_csv(TABLE_DIR / f"table2_pearson.csv")
    n_df.to_csv(TABLE_DIR / f"table2_pearson_n.csv")
    try:
        latex2 = table2_to_latex(corr_df, n_df)
        (TABLE_DIR / f"table2_pearson.tex").write_text(latex2, encoding="utf-8")
        print(f"  [OK] table2_pearson.csv / .tex")
    except ImportError:
        print(f"  [OK] table2_pearson.csv  (LaTeX skipped — scipy not installed)")
        print(f"       pip install scipy to enable significance stars")

    # Table 3
    t3.to_csv(TABLE_DIR / f"table3_trajectory.csv", index=False)
    (TABLE_DIR / f"table3_trajectory.tex").write_text(
        table3_to_latex(t3), encoding="utf-8"
    )
    print(f"  [OK] table3_trajectory.csv / .tex")

    print(f"\n[done] Tables : {TABLE_DIR.resolve()}")
    print(f"[done] Figures: {FIGDIR.resolve()}")
    print("\n[next] Run: python src/05_task_A_prediction.py")


# ──────────────────────────────────────────────────────────────
# Print summary to console
# ──────────────────────────────────────────────────────────────
def print_summary(
    t1: pd.DataFrame,
    corr_df: pd.DataFrame,
) -> None:
    print("\n[step 4] Descriptive summary")
    print(f"\n  {'Indicator':<22} {'Role':>4}  {'Observed%':>10}  "
          f"{'Available%':>11}  {'Missing%':>9}")
    print(f"  {'-'*22} {'-'*4}  {'-'*10}  {'-'*11}  {'-'*9}")
    for _, row in t1.iterrows():
        obs  = f"{row['observed_pct']:.1f}%" if pd.notna(row['observed_pct']) else "  n/a"
        avail = f"{row['available_pct']:.1f}%"
        miss  = f"{row['missing_pct']:.1f}%"
        print(f"  {row['indicator']:<22} {row['role']:>4}  {obs:>10}  "
              f"{avail:>11}  {miss:>9}")

    # Key bivariate correlations for manuscript
    print(f"\n  Key Pearson correlations (complete-case):")
    pairs = [
        ("gpi_sec",         "sec_completion",   "GPI → completion (Task C)"),
        ("edu_spend_gdp",   "sec_completion",   "Spend → completion (Task E)"),
        ("gini",            "sec_completion",   "Gini → completion"),
        ("gdp_pc_ppp",      "sec_completion",   "GDP → completion"),
        ("gov_effect",      "sec_completion",   "Governance → completion"),
        ("gini",            "gpi_sec",          "Gini → GPI (fairness axis)"),
    ]
    for a, b, desc in pairs:
        if a in corr_df.columns and b in corr_df.columns:
            r = corr_df.loc[a, b]
            print(f"  {desc:<38} r = {r:+.3f}" if not np.isnan(r)
                  else f"  {desc:<38} r = n/a")


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    panel, mask, labels = load_data()

    print("\n[step 1] Table 1 — completeness …")
    t1 = build_table1(panel, mask)

    print("[step 2] Table 2 — Pearson correlation …")
    corr_df, n_df = build_table2(panel)

    print("[step 3] Table 3 — trajectory deltas …")
    t3 = build_table3(labels)

    print_summary(t1, corr_df)

    print("\n[step 4] Figures …")
    plot_mnar_chart(panel)
    plot_correlation_heatmap(corr_df, n_df)

    save_tables(t1, corr_df, n_df, t3)


if __name__ == "__main__":
    main()